# Модуль 1. Введение в сбор данных

- [ ] Соберать данные из двух разных источников (открытый датасет + веб-скрейпинг или API).
- [ ] Провести их агрегацию, создав единый датасет.
- [ ] Провести разведывательный анализ данных (EDA).
- [ ]  Постройть базовые визуализации для основных признаков с учетом разметки данных.
- [ ] Описать возможные применения этих данных в контексте машинного обучения.


# Описание датасета
### Идея:

Собрать новостные данные за определенный промежуток времени для получения кластеризации компаний по их наименованиям. Соответственно, решается 2 задачи. Первая - сбор новостных постов, вторая - извлечение названий компаний. В рамках PoC решения будут собраны только новостные посты для их дальнейшей кластеризации.

### Реализация:

  - Первой задачей (решаемая сейчас) будет парсинг новостных Telegram-каналов для получения постов, обогащение датасета открытыми датасетами.
  - Второй задачей (MVP-итерация) - разметка названий упоминаемых компаний для решения NER на новостных постах, обогащение открытыми датасетами

В связи с тем, что асинхронное исполенение кода в Jupyter-notebooks не работает, исполняемый код с парсингом записан в файле `parse.py`. Мы парсим канал "forbesrussia" за последний 21 день и кладем данные в `data/raw/forbes_news.csv`, получили 818 новостных записей. В качестве открытого датасета взяли `yutkin/Lenta.Ru-News-Dataset`, содержащего 800964 новостных записей. Для упрощения себе жизни, будем использовать 1000 последних записей


### Применение для машинного обучения:

Кластеризации компаний по их наименованиям для поиска инфо-поводов.

### Алгоритм обработки:

Объект класса ProductScraper в зависимости от переданного класса магазина, посылает запросы на сайт с помощью Selenium, парсит результаты через BautifulSoup по найденным в коде страницы идентификаторам для поиска нужных элементов. Данные считываются постранично с проверкой существования в csv такого же товара за эту же дату. С сайтов получаем информацию: название товара, цена, скидка, рейтинг. (возможно дальнейшее добавление)

Считанная (новая) информация с текущей датой дозаписывается в csv с названием обрабатываемого магазина.

В дальнейшем производится извлечение дополнительных признаков из поля name:

бренд
вес и единица измерения
в нарезке (да/нет)
БЗМЖ (да/нет)
Далее данные с извлеченными фичами записываются в БД. Обработка пропусков планируется в ЛР 2.

Замечания:

Скрейпинг магазинов потребовал дополнительных шагов: для Пятерочки это отправка предварительного запроса на авторизацию, для Магнита - отправка cookies с адресом магазина.

In [5]:
from pathlib import Path
import os

import pandas as pd


In [6]:
RAW_PATH = Path(r"..\data\raw").as_posix()
PROCESSED_PATH = Path(r"..\data\processed").as_posix()
INTERIM_PATH = Path(r"..\data\interim").as_posix()
EXTERNAL_PATH = Path(r"..\data\external").as_posix()

assert os.path.isdir(RAW_PATH) and os.path.isdir(PROCESSED_PATH) and os.path.isdir(INTERIM_PATH) and os.path.isdir(EXTERNAL_PATH)

# 1. Проведем агрегацию данных и создадим единый датасет


In [ ]:
# Используем последние 1000 записей из датасета Lenta.Ru-News-Dataset, только текст и дату

lenta_df = pd.read_csv(Path(RAW_PATH / Path('lenta-ru-news.csv')), usecols=['date',  'text']).tail(1000)
forbes_df = pd.read_csv(Path(RAW_PATH / Path('forbes_news.csv')), usecols=['date',  'text'])

lenta_df.head()


,text,date
799975,Генеральный директор Первого канала Константин...,2019/12/10
799976,Власти Саудовской Аравии с помощью первичного ...,2019/12/10
799977,Лидер ликвидированного правозащитного движения...,2019/12/10
799978,Пассажирка авиакомпании American Airlines пожа...,2019/12/10
799979,Петербургский «Зенит» крупно проиграл лиссабон...,2019/12/10


In [23]:
forbes_df.head()

,date,text
0,2025-04-08 12:34:04+00:00,"«Мы делаем все, чтобы человек любил рекламу», ..."
1,2025-04-08 12:29:54+00:00,Госдума ратифицировала договор о стратегическо...
2,2025-04-08 12:15:54+00:00,"Госдума приняла закон, разрешающий проводить з..."
3,2025-04-08 11:40:49+00:00,Россию на предстоящей встрече с США в Стамбуле...
4,2025-04-08 11:35:15+00:00,Как основатели банка Credit One заработали мил...


In [29]:
# Приведем даты к формату YYYY-MM-DD
lenta_df['date'] = lenta_df['date'].dt.strftime('%Y-%m-%d')
forbes_df['date'] = forbes_df['date'].dt.strftime('%Y-%m-%d')

lenta_df.head()


,text,date
799975,Генеральный директор Первого канала Константин...,2019-12-10
799976,Власти Саудовской Аравии с помощью первичного ...,2019-12-10
799977,Лидер ликвидированного правозащитного движения...,2019-12-10
799978,Пассажирка авиакомпании American Airlines пожа...,2019-12-10
799979,Петербургский «Зенит» крупно проиграл лиссабон...,2019-12-10


In [30]:
forbes_df.head()

,date,text
0,2025-04-08,"«Мы делаем все, чтобы человек любил рекламу», ..."
1,2025-04-08,Госдума ратифицировала договор о стратегическо...
2,2025-04-08,"Госдума приняла закон, разрешающий проводить з..."
3,2025-04-08,Россию на предстоящей встрече с США в Стамбуле...
4,2025-04-08,Как основатели банка Credit One заработали мил...


In [37]:
# Конкатенируем датасеты и сохраняем в csv

news_df = pd.concat([lenta_df, forbes_df], ignore_index=True)
news_df.to_csv(Path(RAW_PATH / Path('news.csv')), index=False)

In [39]:
news_df.tail()

,text,date
1885,"Путин согласился с предложением Трампа, чтобы ...",2025-03-18
1886,Совет директоров ПАО «Интер РАО» рекомендовал ...,2025-03-18
1887,Телефонный разговор Путина и Трампа завершился...,2025-03-18
1888,В последние годы ESG перестал быть просто модн...,2025-03-18
1889,Google подписала «окончательное соглашение» о ...,2025-03-18


In [ ]:
# 2. Проведем разведывательный анализ данных

# 3. Построим базовые визуализации для основных признаков с учетом разметки данных

# 4. Опишем возможные применения этих данных в контексте машинного обучения


